# Model Evaluation & Business Performance Analysis with Bootstrap

## Overview  
This notebook extends the final evaluation of the trained machine learning models for the Bank Marketing dataset by incorporating statistical uncertainty into the analysis. While previous evaluation focused on point estimates of model performance on the held-out test set, this stage introduces bootstrap resampling to assess the stability and variability of those results.

The evaluation is conducted from both a predictive and a business-oriented perspective. In addition to standard classification metrics, we continue to use a cost–benefit framework that captures the economic impact of marketing decisions. By combining bootstrap resampling with profit-based evaluation, we obtain not only performance estimates but also confidence intervals that reflect the robustness of each modeling strategy.

We compare a global modeling strategy (one model for all clients) against a segmented strategy (cluster-specific models), both at the overall level and within individual clusters, now accounting for variability across repeated samples.

## Objectives  
- Estimate the performance of the global model using bootstrap resampling on the test set
- Estimate the performance of the segmented (cluster-specific) modeling approach under the same framework
- Quantify uncertainty in predictive and business metrics using repeated sampling
- Compare global vs segmented strategies using both mean performance and variability
- Evaluate whether observed differences in expected profit are statistically stable
- Analyze cluster-level performance with uncertainty to assess segmentation impact
- Provide a more robust basis for final decision-making by incorporating variability into model comparison 

## Dataset Description  
This evaluation uses the held-out test dataset generated during the preprocessing phase. The dataset includes engineered features such as categorical encodings, binary indicators, interaction variables, and a cluster assignment for each observation.

The test set remains untouched during model training, calibration, and threshold selection, ensuring that all evaluation results reflect unbiased out-of-sample performance.

Bootstrap resampling is applied directly to this test set, generating multiple resampled datasets of equal size drawn with replacement. This allows us to approximate the sampling distribution of each metric and quantify its variability.

The models evaluated include:
- A global model trained on the full dataset
- Multiple cluster-specific models trained independently per segment

Each model produces calibrated probabilities, which are transformed into binary decisions using thresholds determined during validation.

## Key Considerations  
- **Out-of-sample evaluation with resampling:** All results are based on the test set, with bootstrap resampling used to estimate variability without introducing training bias
- **Consistency with training pipeline:** Feature alignment, calibration, and thresholds are applied exactly as defined during model development
- **Business-oriented evaluation:** Performance is assessed using both classification metrics and expected profit under a cost–benefit framework
- **Threshold-based decision rule:** Predictions are derived using fixed thresholds selected on validation data, ensuring consistency across bootstrap samples
- **Segmentation consistency:** Cluster-specific models are evaluated only within their respective segments
- **Comparability of strategies:** Global and segmented approaches are evaluated under identical assumptions and resampling procedures
- **Uncertainty quantification:** Mean and standard deviation across bootstrap iterations are used to assess the stability and reliability of results 

## Outcome  
By the end of this notebook, we obtain a comprehensive and uncertainty-aware evaluation of the modeling approaches, including:

- Bootstrapped performance estimates (mean and variability) for the global model
- Bootstrapped performance estimates for the segmented (cluster-based) model
- A comparison of strategies that accounts for both average performance and stability
- Cluster-level analysis with uncertainty to identify where segmentation provides consistent benefits
- A consolidated view of expected profit variability and operational impact
- A final assessment of whether segmentation provides robust and reliable business value

These results provide a stronger foundation for selecting a deployment strategy by distinguishing between performance differences that are consistent and those that may be driven by sampling variability.

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.utils import resample
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

from utils.utils import load_dataset, evaluate_with_profit

MODEL_DIR = Path("models")

In [2]:
# quarto preview 05_bootstrap_evaluation.ipynb --to pdf
# quarto render 05_bootstrap_evaluation.ipynb
# black 05_bootstrap_evaluation.ipynb

In [3]:
# Global artifacts
global_model = joblib.load(MODEL_DIR / "global_model.joblib")
global_calibrator = joblib.load(MODEL_DIR / "global_calibrator.joblib")
global_metadata = joblib.load(MODEL_DIR / "global_metadata.joblib")
global_threshold = joblib.load(MODEL_DIR / "global_threshold.joblib")

In [4]:
# Cluster artifacts
cluster_models = {}
cluster_calibrators = {}
cluster_metadata = {}

cluster_thresholds = joblib.load(MODEL_DIR / "cluster_thresholds.joblib")

for cluster_id in cluster_thresholds.keys():
    cluster_models[cluster_id] = joblib.load(MODEL_DIR / f"c{cluster_id}_model.joblib")
    cluster_calibrators[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_calibrator.joblib"
    )
    cluster_metadata[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_metadata.joblib"
    )

In [5]:
X_test_full, y_test = load_dataset("02", "test")

cluster_col = "cluster"

# Keep cluster labels for segmentation
X_test = X_test_full.drop(columns=[cluster_col])


test
X shape: (9043, 60)
y shape: (9043,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0


In [6]:
C_call = 1.0
B_sub = 10.0

## 1. Overall Comparison: Global vs Segmented Model Evaluation

In [7]:
# Align features
X_test_global = X_test[global_metadata["features"]]

# Predict probabilities (raw + calibrated)
global_probs_raw = global_model.predict_proba(X_test_global)[:, 1]
global_probs_cal = global_calibrator.transform(global_probs_raw)

In [8]:
def bootstrap_evaluation(X_full, y_true, global_probs_cal, n_iterations=100):
    """
    Computes bootstrapped metrics for both Global and Segmented strategies,
    including overall performance and per-cluster breakdowns.
    """

    all_results = []

    for i in range(n_iterations):
        # 1. Create bootstrap sample indices
        indices = resample(
            np.arange(len(y_true)), replace=True, n_samples=len(y_true), random_state=i
        )

        y_samp = y_true.iloc[indices]
        X_samp_full = X_full.iloc[indices]

        # --- GLOBAL STRATEGY ---
        probs_g_samp = global_probs_cal[indices]
        metrics_g = evaluate_with_profit(
            y_samp, probs_g_samp, global_threshold, C_call, B_sub
        )

        # --- SEGMENTED STRATEGY ---
        probs_s_samp = np.zeros(len(indices))
        y_pred_s = np.zeros(len(indices))

        for cid in cluster_thresholds.keys():
            m = X_samp_full[cluster_col] == cid
            if not m.any():
                continue

            feat_k = cluster_metadata[cid]["features"]
            X_k = X_samp_full.loc[m, feat_k]

            # Predict and Calibrate
            p_k = cluster_calibrators[cid].transform(
                cluster_models[cid].predict_proba(X_k)[:, 1]
            )
            probs_s_samp[m] = p_k
            y_pred_s[m] = (p_k >= cluster_thresholds[cid]).astype(int)

        # Calculate Segmented Metrics manually to handle multiple thresholds
        metrics_s = {
            "AUC_PR": average_precision_score(y_samp, probs_s_samp),
            "Precision": precision_score(y_samp, y_pred_s, zero_division=0),
            "Recall": recall_score(y_samp, y_pred_s),
            "F1": f1_score(y_samp, y_pred_s),
            "% Contacted": y_pred_s.mean() * 100,
            "Expected Profit": np.sum(probs_s_samp[y_pred_s == 1] * B_sub - C_call),
        }

        # 2. Package all 6 metrics for both models
        row = {"iteration": i}
        for metric in [
            "AUC_PR",
            "Precision",
            "Recall",
            "F1",
            "% Contacted",
            "Expected Profit",
        ]:
            row[f"Global_{metric}"] = metrics_g[metric]
            row[f"Segmented_{metric}"] = metrics_s[metric]

        all_results.append(row)

    return pd.DataFrame(all_results)

In [9]:
# Run bootstrapping
bootstrap_results = bootstrap_evaluation(
    X_test_full, y_test, global_probs_cal, n_iterations=250
)

# Calculate Mean and Std
summary = bootstrap_results.agg(["mean", "std"]).T
summary["Confidence_Interval"] = summary.apply(
    lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1
)

print("Overall Performance with Uncertainty:")
display(summary[["Confidence_Interval"]])

Overall Performance with Uncertainty:


,Confidence_Interval
iteration,124.500 ± 72.313
Global_AUC_PR,0.419 ± 0.016
Segmented_AUC_PR,0.399 ± 0.017
Global_Precision,0.264 ± 0.008
Segmented_Precision,0.272 ± 0.009
Global_Recall,0.713 ± 0.013
Segmented_Recall,0.649 ± 0.014
Global_F1,0.385 ± 0.009
Segmented_F1,0.384 ± 0.010
Global_% Contacted,31.636 ± 0.483


The question that is answered by **Global** is: if I use ONE model for everyone, how do I perform?
The question that is answered by **Segmented** is: if I specialize by cluster, how do I perform overall?

Through the **comparison**, the question we answer is: which strategy wins overall?

After running 250 bootstrapped test samples we get:

The global and segmented strategies show a trade-off between predictive performance and business efficiency. The global model achieves higher recall and slightly better AUC-PR, indicating stronger overall ability to capture positive cases across the full population. In contrast, the segmented approach improves precision while reducing recall, reflecting a more conservative targeting strategy.

From a business perspective, the global model results in higher expected profit under the chosen decision rule, although it also leads to a higher percentage of contacted clients. The segmented strategy reduces contact volume but does not fully translate this reduction into higher profitability at the aggregate level.

Overall, both approaches exhibit similar F1 performance, suggesting comparable balance between precision and recall, but differ primarily in how they trade off coverage versus efficiency.

For all of the metrics, the standard deviation between the Global and Segmented models are pretty similar, almost the same. 

> Across the 250 bootstrap samples, both strategies exhibit nearly identical variability, but the global model consistently maintains higher recall and expected profit, indicating that its advantage is stable rather than driven by sampling noise.

## 2. Per-cluster comparison

In [10]:
# 1. Setup metrics and iterations
n_iterations = 10
cluster_stats = []

for cluster_id in sorted(cluster_thresholds.keys()):
    # Filter the test set for the current cluster
    mask = X_test_full[cluster_col] == cluster_id
    X_k_full = X_test_full[mask]
    y_k = y_test[mask]

    # Probabilities for this cluster only
    probs_g_k = global_probs_cal[mask]  # Global model's view of this cluster

    # For Segmented Model: re-predicting on this cluster's specific features
    feat_k = cluster_metadata[cluster_id]["features"]
    X_k_segmented = X_k_full[feat_k]
    probs_s_k = cluster_calibrators[cluster_id].transform(
        cluster_models[cluster_id].predict_proba(X_k_segmented)[:, 1]
    )

    # 2. Bootstrap within this cluster
    boot_metrics = []
    for i in range(n_iterations):
        idx = resample(
            np.arange(len(y_k)), replace=True, n_samples=len(y_k), random_state=i
        )

        # Slice the cluster data
        y_samp = y_k.iloc[idx]
        p_g_samp = probs_g_k[idx]
        p_s_samp = probs_s_k[idx]

        # Evaluate Global Strategy on this cluster sample
        m_g = evaluate_with_profit(y_samp, p_g_samp, global_threshold, C_call, B_sub)
        m_g["Model"] = "Global"

        # Evaluate Segmented Strategy on this cluster sample
        m_s = evaluate_with_profit(
            y_samp, p_s_samp, cluster_thresholds[cluster_id], C_call, B_sub
        )
        m_s["Model"] = "Segmented"

        boot_metrics.append(m_g)
        boot_metrics.append(m_s)

    # 3. Aggregate results for this cluster
    boot_df_k = pd.DataFrame(boot_metrics)
    for model_type in ["Global", "Segmented"]:
        model_results = boot_df_k[boot_df_k["Model"] == model_type]

        # Calculate mean and std for all metrics
        summary_row = {"Cluster": cluster_id, "Model": model_type}
        for metric in [
            "AUC_PR",
            "Precision",
            "Recall",
            "F1",
            "% Contacted",
            "Expected Profit",
        ]:
            avg = model_results[metric].mean()
            std = model_results[metric].std()

            # Format as Mean ± Std
            if metric == "Expected Profit":
                summary_row[metric] = f"{avg:.0f} ± {std:.0f}"
            elif metric == "% Contacted":
                summary_row[metric] = f"{avg:.2f} ± {std:.2f}"
            else:
                summary_row[metric] = f"{avg:.3f} ± {std:.3f}"

        cluster_stats.append(summary_row)

# 4. Final Display Table
comparison_with_uncertainty_df = pd.DataFrame(cluster_stats)
perf_metrics = [
    "Cluster",
    "Model",
    "AUC_PR",
    "Precision",
    "Recall",
    "F1",
    "% Contacted",
    "Expected Profit",
]
display(comparison_with_uncertainty_df[perf_metrics])

,Cluster,Model,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
0,0,Global,0.469 ± 0.024,0.312 ± 0.018,0.796 ± 0.023,0.448 ± 0.021,35.05 ± 0.84,1312 ± 46
1,0,Segmented,0.429 ± 0.032,0.278 ± 0.019,0.765 ± 0.033,0.408 ± 0.023,37.72 ± 1.31,1316 ± 58
2,1,Global,0.345 ± 0.035,0.266 ± 0.019,0.568 ± 0.037,0.362 ± 0.023,17.85 ± 0.55,801 ± 44
3,1,Segmented,0.299 ± 0.039,0.298 ± 0.029,0.428 ± 0.035,0.351 ± 0.030,12.02 ± 0.58,554 ± 36
4,2,Global,0.428 ± 0.022,0.243 ± 0.013,0.679 ± 0.032,0.358 ± 0.016,31.53 ± 0.56,1135 ± 56
5,2,Segmented,0.393 ± 0.024,0.275 ± 0.013,0.624 ± 0.036,0.382 ± 0.015,25.58 ± 0.81,859 ± 72
6,3,Global,0.444 ± 0.025,0.255 ± 0.015,0.791 ± 0.021,0.385 ± 0.018,44.34 ± 0.93,1894 ± 50
7,3,Segmented,0.433 ± 0.024,0.254 ± 0.016,0.736 ± 0.025,0.377 ± 0.019,41.36 ± 0.62,2089 ± 49


The question that is answered by this is: does segmentation actually improve results WHERE it matters? 

**Machine Learning Metrics**

The per-cluster results highlight meaningful heterogeneity in how the global and segmented strategies perform across different customer segments.

In all clusters, the global model tends to achieve higher recall, indicating stronger ability to identify positive cases, while the segmented models generally improve precision, reflecting more conservative and targeted decision boundaries within each segment.

Performance differences are not uniform across clusters. Some segments show only marginal differences between approaches, while others exhibit more pronounced trade-offs between recall and precision depending on whether a global or specialized model is used. This reinforces the idea that customer behavior and model effectiveness vary across clusters, making segmentation relevant at the local level even if gains are not consistent across all segments.

**Business Oriented Metrics**

At the cluster level, the segmented strategy shows mixed performance relative to the global model, with clear variation across segments.

In some clusters, segmentation improves expected profit, suggesting that specialized models are better aligned with local behavioral patterns and decision boundaries (Cluster 0 and 3). However, in other clusters, the global model performs better (Cluster 1 and 4), indicating that pooling information across the full population can still be beneficial when segment-specific data is more limited or less stable.

The percentage of contacted clients also varies between strategies, reflecting different threshold behaviors per cluster. This leads to differences in operational intensity that are not uniform across segments.

Overall, the results suggest that the value of segmentation is cluster-dependent rather than globally consistent, with some segments benefiting from specialization while others favor a unified modeling approach.

Across clusters, the variability of both machine learning and business metrics is generally comparable between approaches, and in several cases lower for the global model. This suggests that the segmented strategy does not provide more stable or reliable improvements, reinforcing that its benefits are localized rather than consistently robust across segments.